In [ ]:
import pandas as pd
import numpy as np

# Load the CSV with a permissive encoding to handle special characters
df = pd.read_csv('Forwards.csv', encoding='latin1')

# Normalize column names (optional)
df.columns = [c.strip().replace('\\', '').replace('  ', ' ') for c in df.columns]

# Identify Pedro row
mask_pedro = df['shortName'].str.strip().str.lower() == 'pedro'
pedro = df.loc[mask_pedro].squeeze()

# Metrics to analyze (exclude name and Minutes)
metrics = [c for c in df.columns if c not in ['shortName','Minutes']]

# Compute summary stats for each metric
summary = []
for m in metrics:
    series = df[m].astype(float)
    val = float(pedro[m])
    mean = series.mean()
    median = series.median()
    std = series.std(ddof=0)
    # percentile and rank of Pedro within the cohort
    pctl = (series.rank(pct=True, method='average')[mask_pedro].values[0])
    rank = int(series.rank(ascending=False, method='min')[mask_pedro].values[0])
    n = series.shape[0]
    z = (val - mean) / std if std > 0 else np.nan
    summary.append({
        'metric': m,
        'value': val,
        'mean': mean,
        'median': median,
        'std': std,
        'percentile': pctl,
        'rank': rank,
        'n': n,
        'z_score': z
    })

summary_df = pd.DataFrame(summary).sort_values(by='z_score', ascending=False)

# Identify strengths (z > +0.5), average (-0.5 <= z <= +0.5), weaknesses (z < -0.5)
strengths = summary_df[summary_df['z_score'] > 0.5].sort_values('z_score', ascending=False)
average = summary_df[(summary_df['z_score'] >= -0.5) & (summary_df['z_score'] <= 0.5)].sort_values('z_score', ascending=False)
weaknesses = summary_df[summary_df['z_score'] < -0.5].sort_values('z_score')

# Helpers for formatting
def nice_name(s):
    return s.replace('_adjusted_per90','').replace('_',' ').title()

def fmt_pct(p):
    return f"{int(round(p*100))}th percentile"

# Build descriptive sentences (optional)
strength_list = [f"{nice_name(row['metric'])} ({fmt_pct(row['percentile'])}, rank {row['rank']}/{row['n']})"
                 for _, row in strengths.head(3).iterrows()]
weak_list = [f"{nice_name(row['metric'])} ({fmt_pct(row['percentile'])}, rank {row['rank']}/{row['n']})"
             for _, row in weaknesses.head(3).iterrows()]
avg_list = [f"{nice_name(row['metric'])} (~{fmt_pct(row['percentile'])})"
            for _, row in average.sort_values('percentile').head(2).iterrows()]

strength_sentence = "Pedro’s standout strengths are " + ", ".join(strength_list) + "."
weak_sentence = "He is weaker in " + ", ".join(weak_list) + (", and around average in " + ", ".join(avg_list) if avg_list else "") + "."

# Concluding statement
conclusion = (
    f"Overall, Pedro profiles as a high-volume final-third contributor (passes {pedro['final_third_passes_adjusted_per90']:.1f}/90, "
    f"receptions {pedro['final_third_receptions_adjusted_per90']:.1f}/90) who adds steady goal threat "
    f"({pedro['goals_adjusted_per90']:.2f}/90) and creative output ({pedro['key_passes_adjusted_per90']:.2f} key passes & "
    f"{pedro['assists_adjusted_per90']:.2f} assists/90) over {int(pedro['Minutes'])} minutes, best deployed as an advanced "
    f"wide forward/creator rather than a target man."
)

# If you want to inspect the tables:
top_strengths_table = strengths[['metric','value','percentile','rank','n','z_score']].head(5)
top_weaknesses_table = weaknesses[['metric','value','percentile','rank','n','z_score']].head(5)

print(strength_sentence)
print(weak_sentence)
print(conclusion)